<a href="https://colab.research.google.com/github/AssylZh/capstone/blob/main/codes_for_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
SVR + Gradient Boosting + Random Forest forecast of next-year real median wage growth, Kazakhstan regions.
Usage:  python svr_wage_growth.py extended_data_5.xlsx
Needs:  pandas numpy scikit-learn openpyxl
"""
import sys
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.linear_model import LinearRegression

PATH = "extended data (6).xlsx"
if len(sys.argv) > 1 and sys.argv[1].lower().endswith((".xlsx", ".xls")):
    PATH = sys.argv[1]
TEST_YEARS = [2023, 2024, 2025]      # forecast origins (target years)
FIRST_WAGE_YEAR = 2018               # 2017 medians look unreliable
NATIONAL = "Republic of Kazakhstan"

# ------------------------------------------------------------------ 1. data
def load_panel(path):
    """Read the sheet that holds the panel (the one with 'region' and 'year' columns),
    whatever it is called; skips README sheets."""
    book = pd.read_excel(path, sheet_name=None)
    for name, sheet in book.items():
        sheet.columns = [str(c).strip() for c in sheet.columns]
        if {"region", "year", "wage_median"} <= set(sheet.columns):
            print(f"Using sheet: {name!r}")
            return sheet
    raise ValueError(f"No sheet with region/year/wage_median columns. "
                     f"Sheets found: {list(book)}")

raw = load_panel(PATH)
for c in ["wage_median", "cpi_index", "unemployment_rate_pct",
          "grp_mln_tenge", "population"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

# National CPI = the national row's cpi_index, applied to every region
# (the regional cpi_index column is patchy and on different bases).
cpi = (raw[raw.region == NATIONAL].set_index("year")["cpi_index"]
       .rename("cpi_nat"))

df = raw[(raw.region != NATIONAL) &
         (~raw.region.str.contains("historical")) &
         (raw.year >= FIRST_WAGE_YEAR)].copy()
df = df.merge(cpi, left_on="year", right_index=True, how="left")
df = df.sort_values(["region", "year"])

df["real_wage"] = df.wage_median / (df.cpi_nat / 100)
df["grp_pc"] = df.grp_mln_tenge * 1e6 / df.population
g = df.groupby("region")
df["y"] = g.real_wage.pct_change()                 # real wage growth in year t
df["nom_g"] = g.wage_median.pct_change()
df["grp_g"] = g.grp_pc.pct_change()
df["pop_g"] = g.population.pct_change()
nat = (raw[raw.region == NATIONAL].set_index("year").wage_median
       .div(cpi / 100).pct_change().rename("nat_g"))   # national real growth

# Features are LAGGED one year, so nothing from year t leaks into the forecast
g = df.groupby("region")
X = pd.DataFrame({
    "region": df.region, "year": df.year, "y": df.y,
    "y_lag1": g.y.shift(1),
    "nom_g_lag1": g.nom_g.shift(1),
    "unemp_lag1": g.unemployment_rate_pct.shift(1),
    "grp_g_lag1": g.grp_g.shift(1),
    "pop_g_lag1": g.pop_g.shift(1),
}).merge(nat.shift(1).rename("nat_g_lag1"), left_on="year", right_index=True, how="left")
X["gap_lag1"] = X.y_lag1 - X.nat_g_lag1            # last year's gap to national
X["nat_g"] = X.year.map(nat)                       # national growth in year t (for the flag)
FEATS = ["y_lag1", "nom_g_lag1", "unemp_lag1", "grp_g_lag1",
         "pop_g_lag1", "nat_g_lag1", "gap_lag1"]
X = X.dropna(subset=["y"] + FEATS).reset_index(drop=True)
print(f"Usable rows: {len(X)}, regions: {X.region.nunique()}, "
      f"target years: {sorted(X.year.unique())}")

# ------------------------------------------------------------------ 2. model
def tune(pipe, grid, train):
    """Pick settings using ONLY training years: the last 2 training years are
    held out for validation (time-ordered, so no peeking at the future)."""
    years = sorted(train.year.unique())
    if len(years) >= 3:
        fold = np.where(train.year.isin(years[-2:]), train.year, -1)
        cv = PredefinedSplit(fold)
    else:
        cv = 2
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="neg_root_mean_squared_error")
    gs.fit(train[FEATS], train.y)
    return gs.best_estimator_, gs.best_params_

def fit_svr(train):
    pipe = make_pipeline(StandardScaler(), SVR(kernel="rbf"))
    grid = {"svr__C": [0.1, 1, 10],
            "svr__epsilon": [0.005, 0.01, 0.02],
            "svr__gamma": ["scale", 0.05, 0.2]}
    return tune(pipe, grid, train)

def fit_gb(train):
    """Gradient boosting = many small decision trees added one after another,
    each one correcting the mistakes of the trees before it.
    Settings are kept small/shallow because we only have ~50-80 training rows."""
    pipe = GradientBoostingRegressor(subsample=0.8, random_state=0)
    grid = {"n_estimators": [50, 150],
            "learning_rate": [0.03, 0.1],
            "max_depth": [1, 2, 3],
            "min_samples_leaf": [3, 6]}
    return tune(pipe, grid, train)

def fit_rf(train):
    """Random forest = many decision trees, each grown on a random resample of the
    rows and a random subset of clues, then averaged. Averaging many different trees
    makes the forecast stable. Kept small because we only have ~50-80 training rows."""
    pipe = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)
    grid = {"max_depth": [2, 3, 5, None],
            "min_samples_leaf": [3, 6, 10],
            "max_features": [0.33, 0.6, 1.0]}
    return tune(pipe, grid, train)

# ------------------------------------------------------------------ 3. rolling origin
rows, importances, importances_rf = [], [], []
for T in TEST_YEARS:
    train, test = X[X.year < T], X[X.year == T].copy()
    if len(train) < 20 or test.empty:
        continue
    svr, best = fit_svr(train)
    gb, best_gb = fit_gb(train)
    rf, best_rf = fit_rf(train)
    print(f"{T}: n_train={len(train)}, n_test={len(test)}")
    print(f"     SVR best={best}")
    print(f"     GB  best={best_gb}")
    print(f"     RF  best={best_rf}")
    # benchmarks
    ar = LinearRegression().fit(train[["y_lag1"]], train.y)   # pooled AR(1)
    test["svr"] = svr.predict(test[FEATS])
    test["gb"] = gb.predict(test[FEATS])
    test["rf"] = rf.predict(test[FEATS])
    importances_rf.append(pd.Series(rf.feature_importances_, index=FEATS).rename(T))
    imp = pd.Series(gb.feature_importances_, index=FEATS).rename(T)
    importances.append(imp)
    test["ar1"] = ar.predict(test[["y_lag1"]])
    test["rw_last"] = test.y_lag1                              # repeat last growth
    test["rw_nat"] = test.nat_g_lag1                           # last national growth
    test["mean_train"] = train.y.mean()
    rows.append(test)
res = pd.concat(rows, ignore_index=True)
MODELS = ["svr", "gb", "rf", "ar1", "rw_last", "rw_nat", "mean_train"]

# ------------------------------------------------------------------ 4. results
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
mae = lambda e: float(np.mean(np.abs(e)))
tab = pd.DataFrame({m: {"RMSE": rmse(res.y - res[m]),
                        "MAE": mae(res.y - res[m])} for m in MODELS}).T
tab["RMSE / RMSE(AR1)"] = tab.RMSE / tab.loc["ar1", "RMSE"]
print("\nPooled out-of-sample accuracy, years", TEST_YEARS)
print(tab.round(4))

by_year = res.groupby("year").apply(
    lambda d: pd.Series({m: rmse(d.y - d[m]) for m in MODELS}), include_groups=False)
by_year["SVR/AR1"] = by_year.svr / by_year.ar1
by_year["GB/AR1"] = by_year.gb / by_year.ar1
by_year["RF/AR1"] = by_year.rf / by_year.ar1
print("\nRMSE by test year"); print(by_year.round(4))

by_reg = res.groupby("region").apply(
    lambda d: pd.Series({"SVR_RMSE": rmse(d.y - d.svr),
                         "GB_RMSE": rmse(d.y - d.gb),
                         "RF_RMSE": rmse(d.y - d.rf),
                         "AR1_RMSE": rmse(d.y - d.ar1),
                         "SVR_bias": float((d.svr - d.y).mean()),
                         "GB_bias": float((d.gb - d.y).mean()),
                         "RF_bias": float((d.rf - d.y).mean()),
                         "n_forecasts": len(d)}), include_groups=False)
by_reg["SVR/AR1"] = by_reg.SVR_RMSE / by_reg.AR1_RMSE
by_reg["GB/AR1"] = by_reg.GB_RMSE / by_reg.AR1_RMSE
by_reg["RF/AR1"] = by_reg.RF_RMSE / by_reg.AR1_RMSE
print("\nBy region (bias > 0 = model over-predicts)"); print(by_reg.round(4))

# below-national-average flag
res["actual_below"] = (res.y < res.nat_g).astype(int)
res["pred_below"] = (res.svr < res.nat_g).astype(int)
res["pred_below_gb"] = (res.gb < res.nat_g).astype(int)
res["pred_below_rf"] = (res.rf < res.nat_g).astype(int)
print("\nBelow-national-average flag accuracy "
      f"(base rate of 'below': {res.actual_below.mean():.2f})")
print(f"  SVR: {(res.actual_below == res.pred_below).mean():.2f}")
print(f"  GB : {(res.actual_below == res.pred_below_gb).mean():.2f}")
print(f"  RF : {(res.actual_below == res.pred_below_rf).mean():.2f}")

imp = pd.concat(importances, axis=1)
imp["average"] = imp.mean(axis=1)
print("\nGradient boosting: which clues mattered (share of importance, per test year)")
print(imp.sort_values("average", ascending=False).round(3))
imp.to_csv("gb_feature_importance.csv")
imp_rf = pd.concat(importances_rf, axis=1)
imp_rf["average"] = imp_rf.mean(axis=1)
print("\nRandom forest: which clues mattered")
print(imp_rf.sort_values("average", ascending=False).round(3))
imp_rf.to_csv("rf_feature_importance.csv")

res.to_csv("model_predictions.csv", index=False)
by_reg.to_csv("model_by_region.csv")
print("\nSaved model_predictions.csv, model_by_region.csv, gb_/rf_feature_importance.csv")

Using sheet: 'Master_Panel'
Usable rows: 103, regions: 20, target years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
2023: n_train=49, n_test=17
     SVR best={'svr__C': 1, 'svr__epsilon': 0.01, 'svr__gamma': 0.2}
     GB  best={'learning_rate': 0.03, 'max_depth': 1, 'min_samples_leaf': 3, 'n_estimators': 50}
     RF  best={'max_depth': 5, 'max_features': 0.33, 'min_samples_leaf': 3}
2024: n_train=66, n_test=17
     SVR best={'svr__C': 1, 'svr__epsilon': 0.02, 'svr__gamma': 0.2}
     GB  best={'learning_rate': 0.1, 'max_depth': 2, 'min_samples_leaf': 3, 'n_estimators': 50}
     RF  best={'max_depth': 5, 'max_features': 1.0, 'min_samples_leaf': 3}
2025: n_train=83, n_test=20
     SVR best={'svr__C': 0.1, 'svr__epsilon': 0.01, 'svr__gamma': 'scale'}
     GB  best={'learning_rate': 0.03, 'max_depth': 3, 'min_samples_leaf': 3, 'n_estimators': 150}
     RF  best={'max_depth': 5, 'max_features': 1.0, 'min_samples_leaf': 3}

Pooled out-of-s

In [3]:
"""National vs regional CPI comparison. Runs BOTH versions and prints the comparison table.
Needs: pandas numpy scikit-learn openpyxl."""
import os, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.linear_model import LinearRegression

PATH = "extended_data_6.xlsx"
if not os.path.exists(PATH):
    found = glob.glob("*extended*data*6*.xlsx") + glob.glob("/content/*extended*data*6*.xlsx")
    if not found:
        raise FileNotFoundError("Upload extended_data_6.xlsx (Colab Files panel) and set PATH above.")
    PATH = found[0]
print("Using", PATH)

TEST_YEARS = [2023, 2024, 2025]; FIRST = 2018; NATIONAL = "Republic of Kazakhstan"
raw0 = pd.read_excel(PATH, sheet_name="Master_Panel")
for c in ["wage_median", "cpi_index", "unemployment_rate_pct", "grp_mln_tenge", "population"]:
    raw0[c] = pd.to_numeric(raw0[c], errors="coerce")
M = ["svr", "gb", "rf", "ar1", "rw_last", "rw_nat", "mean_train"]
FEATS = ["y_lag1", "nom_g_lag1", "unemp_lag1", "grp_g_lag1", "pop_g_lag1", "nat_g_lag1", "gap_lag1"]

def tune(pipe, grid, train):
    yrs = sorted(train.year.unique())
    cv = PredefinedSplit(np.where(train.year.isin(yrs[-2:]), train.year, -1)) if len(yrs) >= 3 else 2
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="neg_root_mean_squared_error"); gs.fit(train[FEATS], train.y)
    return gs.best_estimator_
def fit_svr(t): return tune(make_pipeline(StandardScaler(), SVR(kernel="rbf")),
    {"svr__C": [0.1, 1, 10], "svr__epsilon": [0.005, 0.01, 0.02], "svr__gamma": ["scale", 0.05, 0.2]}, t)
def fit_gb(t): return tune(GradientBoostingRegressor(subsample=0.8, random_state=0),
    {"n_estimators": [50, 150], "learning_rate": [0.03, 0.1], "max_depth": [1, 2, 3], "min_samples_leaf": [3, 6]}, t)
def fit_rf(t): return tune(RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
    {"max_depth": [2, 3, 5, None], "min_samples_leaf": [3, 6, 10], "max_features": [0.33, 0.6, 1.0]}, t)

def run(MODE):
    raw = raw0.copy()
    cpi_nat = raw[raw.region == NATIONAL].set_index("year")["cpi_index"].rename("cpi_nat")
    df = raw[(raw.region != NATIONAL) & (~raw.region.str.contains("historical")) & (raw.year >= FIRST)].copy()
    df = df.merge(cpi_nat, left_on="year", right_index=True, how="left").sort_values(["region", "year"])
    deflator = df.cpi_nat if MODE == "national" else df.cpi_index
    df["real_wage"] = df.wage_median / (deflator / 100)
    df["grp_pc"] = df.grp_mln_tenge * 1e6 / df.population
    pc = lambda s: s.pct_change(fill_method=None)
    g = df.groupby("region")
    df["y"] = g.real_wage.transform(pc); df["nom_g"] = g.wage_median.transform(pc)
    df["grp_g"] = g.grp_pc.transform(pc); df["pop_g"] = g.population.transform(pc)
    nat = (raw[raw.region == NATIONAL].set_index("year").wage_median.div(cpi_nat / 100)
           ).pct_change(fill_method=None).rename("nat_g")
    g = df.groupby("region")
    X = pd.DataFrame({"region": df.region, "year": df.year, "y": df.y, "y_lag1": g.y.shift(1),
        "nom_g_lag1": g.nom_g.shift(1), "unemp_lag1": g.unemployment_rate_pct.shift(1),
        "grp_g_lag1": g.grp_g.shift(1), "pop_g_lag1": g.pop_g.shift(1)}).merge(
        nat.shift(1).rename("nat_g_lag1"), left_on="year", right_index=True, how="left")
    X["gap_lag1"] = X.y_lag1 - X.nat_g_lag1; X["nat_g"] = X.year.map(nat)
    X = X.dropna(subset=["y"] + FEATS).reset_index(drop=True)
    print(f"\n[{MODE}] usable rows {len(X)}, regions {X.region.nunique()}")
    rows = []
    for T in TEST_YEARS:
        tr, te = X[X.year < T], X[X.year == T].copy()
        if len(tr) < 20 or te.empty: continue
        s, gb, rf = fit_svr(tr), fit_gb(tr), fit_rf(tr)
        ar = LinearRegression().fit(tr[["y_lag1"]], tr.y)
        te["svr"] = s.predict(te[FEATS]); te["gb"] = gb.predict(te[FEATS]); te["rf"] = rf.predict(te[FEATS])
        te["ar1"] = ar.predict(te[["y_lag1"]]); te["rw_last"] = te.y_lag1
        te["rw_nat"] = te.nat_g_lag1; te["mean_train"] = tr.y.mean()
        rows.append(te); print(f"  {T}: train {len(tr)}, test {len(te)}")
    res = pd.concat(rows, ignore_index=True)
    res.to_csv(f"predictions_{MODE}.csv", index=False)
    return res

def table(d):
    t = pd.DataFrame({m: {"RMSE": np.sqrt(((d.y - d[m]) ** 2).mean()),
                          "MAE": (d.y - d[m]).abs().mean(), "bias": (d[m] - d.y).mean()} for m in M}).T
    t["ratio_to_AR1"] = t.RMSE / t.loc["ar1", "RMSE"]
    return t.round(4)

nat_res = run("national")
reg_res = run("regional")
same = nat_res.merge(reg_res[["region", "year"]], on=["region", "year"])   # national run on the regional sample
print("\n=== NATIONAL CPI, all", len(nat_res), "forecasts ===");           print(table(nat_res))
print("\n=== NATIONAL CPI, same", len(same), "region-years as regional ==="); print(table(same))
print("\n=== REGIONAL CPI,", len(reg_res), "forecasts ===");                print(table(reg_res))

Using extended data (6).xlsx

[national] usable rows 103, regions 20
  2023: train 49, test 17
  2024: train 66, test 17
  2025: train 83, test 20

[regional] usable rows 69, regions 15
  2023: train 37, test 11
  2024: train 48, test 11
  2025: train 59, test 10

=== NATIONAL CPI, all 54 forecasts ===
              RMSE     MAE    bias  ratio_to_AR1
svr         0.0790  0.0628  0.0426        0.9963
gb          0.0755  0.0595  0.0474        0.9523
rf          0.0723  0.0569  0.0432        0.9115
ar1         0.0793  0.0676  0.0622        1.0000
rw_last     0.0744  0.0562  0.0236        0.9382
rw_nat      0.0514  0.0419  0.0238        0.6479
mean_train  0.0719  0.0612  0.0542        0.9062

=== NATIONAL CPI, same 32 region-years as regional ===
              RMSE     MAE    bias  ratio_to_AR1
svr         0.0682  0.0537  0.0326        0.9393
gb          0.0636  0.0497  0.0384        0.8756
rf          0.0625  0.0480  0.0354        0.8607
ar1         0.0726  0.0606  0.0563        1.0000
rw_

In [4]:
# Figure 1 (overall RMSE) and Figure 2 (RMSE by test year). Numbers are typed from the results tables.
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
blue='#2b6fcf'; grey='#b9b9ae'
rows=[('Last national growth',.0514,0),('Training mean',.0719,0),('Random forest',.0723,1),('Last own growth',.0744,0),('Gradient boosting',.0755,1),('Support vector regression',.0790,1),('AR(1)',.0793,0)]
fig,ax=plt.subplots(figsize=(7,3.2))
for i,(n,v,ml) in enumerate(rows[::-1]):
    ax.barh(i,v,color=blue if ml else grey,height=.65); ax.text(v+.001,i,f'{v:.4f} ({v/.0793:.2f})',va='center',fontsize=8)
ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows[::-1]],fontsize=9)
ax.axvline(.0793,ls='--',color='#666',lw=1); ax.set_xlim(0,.105); ax.set_xlabel('Out-of-sample RMSE (growth units)',fontsize=9)
for s in ('top','right'): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig('fig_rmse.png',dpi=200)
S={'SVR':[.0467,.0473,.1145],'Gradient boosting':[.0490,.0457,.1076],'Random forest':[.0474,.0431,.1031],'AR(1)':[.0606,.0676,.0999],'Last national growth':[.0460,.0537,.0537]}
cols={'SVR':'#9dbbe8','Gradient boosting':'#5e92da','Random forest':blue,'AR(1)':grey,'Last national growth':'#555555'}
fig,ax=plt.subplots(figsize=(7,3.2)); w=.15
for k,(n,v) in enumerate(S.items()):
    xs=[i+(k-2)*w for i in range(3)]; ax.bar(xs,[a*100 for a in v],w*.92,label=n,color=cols[n])
    for x,a in zip(xs,v): ax.text(x,a*100+.15,f'{a*100:.1f}',ha='center',fontsize=6.5)
ax.set_xticks(range(3)); ax.set_xticklabels([2023,2024,2025]); ax.set_ylabel('RMSE (percentage points)',fontsize=9)
ax.legend(fontsize=7,ncol=3,frameon=False,loc='upper left'); ax.set_ylim(0,14)
for s in ('top','right'): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig('fig_year.png',dpi=200)

In [5]:
import pandas as pd, matplotlib, matplotlib.ticker; matplotlib.use("Agg"); import matplotlib.pyplot as plt
reg=pd.read_csv("model_by_region.csv").set_index("region")
pred=pd.read_csv("model_predictions.csv")
# Figure 3: RMSE ratio to AR(1) by region (below 1 = beats AR(1))
reg=reg.sort_values("RF/AR1")
fig,ax=plt.subplots(figsize=(6.5,6))
for col,lab,c,m in [("SVR/AR1","SVR","#9dbbe8","s"),("GB/AR1","Gradient boosting","#5e92da","^"),("RF/AR1","Random forest","#2b6fcf","o")]:
    ax.scatter(reg[col],range(len(reg)),label=lab,color=c,marker=m,s=28)
ax.axvline(1,ls="--",color="#555",lw=1)
ax.set_yticks(range(len(reg))); ax.set_yticklabels(reg.index,fontsize=8)
ax.set_xscale("log"); ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter()); ax.set_xticks([.5,1,2,3]); ax.set_xticklabels(["0.5","1","2","3"])
ax.set_xlabel("RMSE ratio to AR(1), log scale (below 1 beats AR(1))",fontsize=9); ax.legend(fontsize=8,frameon=False)
for s in ("top","right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig("fig_region_ratio.png",dpi=200)
# Figure 4: forecast against actual growth
fig,axs=plt.subplots(1,3,figsize=(9,3.3),sharex=True,sharey=True)
lo,hi=-0.08,0.20
for ax,(m,t) in zip(axs,[("svr","SVR"),("gb","Gradient boosting"),("rf","Random forest")]):
    for y,c in [(2023,"#b9b9ae"),(2024,"#5e92da"),(2025,"#c0392b")]:
        d=pred[pred.year==y]; ax.scatter(d.y*100,d[m]*100,s=14,color=c,label=str(y))
    ax.plot([lo*100,hi*100],[lo*100,hi*100],color="#555",lw=1,ls="--"); ax.set_title(t,fontsize=9); ax.set_xlim(lo*100,hi*100); ax.set_ylim(lo*100,hi*100); ax.set_aspect("equal")
    ax.set_xlabel("Actual growth (%)",fontsize=8)
axs[0].set_ylabel("Forecast growth (%)",fontsize=8); axs[0].legend(fontsize=7,frameon=False)
for ax in axs:
    for s in ("top","right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig("fig_forecast_actual.png",dpi=200)

In [6]:
"""Improvements 1 and 2: (1) forecast each region's GAP to national real wage growth instead of its raw growth,
(2) add inflation to the predictors. Compares four versions on the same 2023-2025 test years:
  V0  raw growth target, original predictors            (the earlier results)
  V1  raw growth target, + inflation predictors          (improvement 2 only)
  V2  gap target, original predictors                    (improvement 1 only)
  V3  gap target, relative predictors + inflation        (improvements 1 and 2)
For gap versions the region's growth forecast = last year's national growth + predicted gap.
Needs: pandas numpy scikit-learn scipy openpyxl."""
import os, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from scipy.stats import spearmanr
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.linear_model import LinearRegression

PATH = "extended_data_6.xlsx"
if not os.path.exists(PATH):
    found = glob.glob("*extended*data*6*.xlsx") + glob.glob("/content/*extended*data*6*.xlsx")
    if not found: raise FileNotFoundError("Upload extended_data_6.xlsx and set PATH above.")
    PATH = found[0]
TEST_YEARS, FIRST, NATIONAL = [2023, 2024, 2025], 2018, "Republic of Kazakhstan"

raw = pd.read_excel(PATH, sheet_name="Master_Panel")
for c in ["wage_median", "cpi_index", "unemployment_rate_pct", "grp_mln_tenge", "population"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
cpi = raw[raw.region == NATIONAL].set_index("year")["cpi_index"]
natw = raw[raw.region == NATIONAL].set_index("year").wage_median
pc = lambda s: s.pct_change(fill_method=None)
nat_g = pc(natw.div(cpi / 100)); nat_nom = pc(natw); infl = pc(cpi)

df = raw[(raw.region != NATIONAL) & (~raw.region.str.contains("historical")) &
         (raw.year >= FIRST)].sort_values(["region", "year"]).copy()
df["real_wage"] = df.wage_median / (df.year.map(cpi) / 100)
df["grp_pc"] = df.grp_mln_tenge * 1e6 / df.population
g = df.groupby("region")
df["y"] = g.real_wage.transform(pc); df["nom_g"] = g.wage_median.transform(pc)
df["grp_g"] = g.grp_pc.transform(pc); df["pop_g"] = g.population.transform(pc)
df["nat_g"] = df.year.map(nat_g); df["gap"] = df.y - df.nat_g
# relative (region minus national / minus other regions) versions of the clues
df["nom_gap"] = df.nom_g - df.year.map(nat_nom)
df["grp_gap"] = df.grp_g - df.groupby("year").grp_g.transform("mean")
df["pop_gap"] = df.pop_g - df.groupby("year").pop_g.transform("mean")
df["unemp_dev"] = df.unemployment_rate_pct - df.groupby("year").unemployment_rate_pct.transform("mean")
df["infl"] = df.year.map(infl); df["nat_nom"] = df.year.map(nat_nom)
g = df.groupby("region")
lag = lambda c: g[c].shift(1)
X = pd.DataFrame({"region": df.region, "year": df.year, "y": df.y, "nat_g_t": df.nat_g, "gap_t": df.gap,
    "y_lag1": lag("y"), "nom_g_lag1": lag("nom_g"), "unemp_lag1": lag("unemployment_rate_pct"),
    "grp_g_lag1": lag("grp_g"), "pop_g_lag1": lag("pop_g"), "nat_g_lag1": lag("nat_g"), "gap_lag1": lag("gap"),
    "nom_gap_lag1": lag("nom_gap"), "grp_gap_lag1": lag("grp_gap"), "pop_gap_lag1": lag("pop_gap"),
    "unemp_dev_lag1": lag("unemp_dev"), "infl_lag1": lag("infl"), "nat_nom_lag1": lag("nat_nom")})
BASE = ["y_lag1", "nom_g_lag1", "unemp_lag1", "grp_g_lag1", "pop_g_lag1", "nat_g_lag1", "gap_lag1"]
INFL = BASE + ["infl_lag1", "nat_nom_lag1"]
REL = ["gap_lag1", "nom_gap_lag1", "grp_gap_lag1", "pop_gap_lag1", "unemp_dev_lag1", "nat_g_lag1", "infl_lag1", "nat_nom_lag1"]
VARIANTS = {"V0": ("y", BASE), "V1": ("y", INFL), "V2": ("gap_t", BASE), "V3": ("gap_t", REL)}
ALLF = sorted(set(BASE + INFL + REL))
X = X.dropna(subset=["y"] + ALLF).reset_index(drop=True)          # identical rows for every version
print(f"Usable rows: {len(X)}, regions: {X.region.nunique()}")

def tune(pipe, grid, train, feats, target):
    yrs = sorted(train.year.unique())
    cv = PredefinedSplit(np.where(train.year.isin(yrs[-2:]), train.year, -1))
    return GridSearchCV(pipe, grid, cv=cv, scoring="neg_root_mean_squared_error").fit(train[feats], train[target]).best_estimator_
MODELS = {
 "svr": (make_pipeline(StandardScaler(), SVR(kernel="rbf")),
         {"svr__C": [0.1, 1, 10], "svr__epsilon": [0.005, 0.01, 0.02], "svr__gamma": ["scale", 0.05, 0.2]}),
 "gb": (GradientBoostingRegressor(subsample=0.8, random_state=0),
        {"n_estimators": [50, 150], "learning_rate": [0.03, 0.1], "max_depth": [1, 2, 3], "min_samples_leaf": [3, 6]}),
 "rf": (RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
        {"max_depth": [2, 3, 5, None], "min_samples_leaf": [3, 6, 10], "max_features": [0.33, 0.6, 1.0]})}

rows = []
for T in TEST_YEARS:
    tr, te = X[X.year < T], X[X.year == T].copy()
    ar = LinearRegression().fit(tr[["y_lag1"]], tr.y)
    te["ar1"] = ar.predict(te[["y_lag1"]]); te["rw_nat"] = te.nat_g_lag1
    te["gap_zero"] = 0.0; te["gap_last"] = te.gap_lag1; te["gap_mean"] = tr.gap_t.mean()
    for v, (target, feats) in VARIANTS.items():
        for m, (pipe, grid) in MODELS.items():
            p = tune(pipe, grid, tr, feats, target).predict(te[feats])
            if target == "y":
                te[f"{v}_{m}"] = p; te[f"{v}_{m}_gap"] = p - te.nat_g_lag1     # implied gap vs last national growth
            else:
                te[f"{v}_{m}_gap"] = p; te[f"{v}_{m}"] = te.nat_g_lag1 + p      # growth = last national + predicted gap
    rows.append(te); print("done", T)
res = pd.concat(rows, ignore_index=True)
res.to_csv("improvement_predictions.csv", index=False)

rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
print("\n=== A. Region growth: RMSE on all 54 forecasts (ratio to AR(1)); excl. 2025; 2025 only ===")
ar_all = rmse(res.y - res.ar1); rows_out = []
for name, col in [("AR(1)", "ar1"), ("last national growth", "rw_nat")] + \
        [(f"{v} {m}", f"{v}_{m}") for v in VARIANTS for m in MODELS]:
    e = res.y - res[col]; e25 = e[res.year == 2025]; e2324 = e[res.year != 2025]
    rows_out.append({"model": name, "RMSE": rmse(e), "ratio_to_AR1": rmse(e) / ar_all,
                     "RMSE_excl_2025": rmse(e2324), "RMSE_2025": rmse(e25), "bias": float((res[col] - res.y).mean())})
print(pd.DataFrame(rows_out).round(4).to_string(index=False))

print("\n=== B. The regional question: forecast the GAP to national growth (actual national growth known) ===")
res["gap_actual"] = res.y - res.nat_g_t
rows_out = []
for name, col in [("zero gap (every region = national)", "gap_zero"), ("last own gap", "gap_last"), ("training-mean gap", "gap_mean")] + \
        [(f"{v} {m}", f"{v}_{m}_gap") for v in VARIANTS for m in MODELS]:
    e = res.gap_actual - res[col]
    rho = np.nanmean([spearmanr(d.gap_actual, d[col])[0] if d[col].nunique() > 1 else np.nan for _, d in res.groupby("year")])
    flag = ((res[col] < 0) == (res.gap_actual < 0)).mean()
    rows_out.append({"model": name, "gap_RMSE": rmse(e), "rank_corr_within_year": rho, "below_avg_accuracy": flag})
print(f"(share of region-years truly below national: {(res.gap_actual < 0).mean():.2f})")
print(pd.DataFrame(rows_out).round(3).to_string(index=False))

try:
    from google.colab import files
    files.download("improvement_predictions.csv")
except ImportError:
    print("\nSaved improvement_predictions.csv in the current folder.")

Usable rows: 103, regions: 20
done 2023
done 2024
done 2025

=== A. Region growth: RMSE on all 54 forecasts (ratio to AR(1)); excl. 2025; 2025 only ===
               model   RMSE  ratio_to_AR1  RMSE_excl_2025  RMSE_2025    bias
               AR(1) 0.0793        1.0000          0.0642     0.0999  0.0622
last national growth 0.0514        0.6479          0.0500     0.0537  0.0238
              V0 svr 0.0790        0.9963          0.0470     0.1145  0.0426
               V0 gb 0.0755        0.9523          0.0474     0.1076  0.0474
               V0 rf 0.0723        0.9115          0.0453     0.1031  0.0432
              V1 svr 0.0777        0.9795          0.0480     0.1112  0.0410
               V1 gb 0.0758        0.9554          0.0474     0.1081  0.0495
               V1 rf 0.0718        0.9055          0.0448     0.1025  0.0417
              V2 svr 0.0785        0.9905          0.0884     0.0582 -0.0017
               V2 gb 0.0499        0.6289          0.0477     0.0533  0.0188
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
"""Forecast next year's (2026) real median wage growth for each region with SVR, gradient boosting and
random forest, trained on every year up to 2025.
Needs: pandas numpy scikit-learn openpyxl"""
import os, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.linear_model import LinearRegression

PATH = "extended_data_6.xlsx"
if not os.path.exists(PATH):
    found = glob.glob("*extended*data*6*.xlsx") + glob.glob("/content/*extended*data*6*.xlsx")
    if not found: raise FileNotFoundError("Upload extended_data_6.xlsx and set PATH above.")
    PATH = found[0]
FIRST, LAST, TARGET = 2018, 2025, 2026
NATIONAL = "Republic of Kazakhstan"
FEATS = ["y_lag1", "nom_g_lag1", "unemp_lag1", "grp_g_lag1", "pop_g_lag1", "nat_g_lag1", "gap_lag1"]

raw = pd.read_excel(PATH, sheet_name="Master_Panel")
for c in ["wage_median", "cpi_index", "unemployment_rate_pct", "grp_mln_tenge", "population"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
cpi_nat = raw[raw.region == NATIONAL].set_index("year")["cpi_index"]
df = raw[(raw.region != NATIONAL) & (~raw.region.str.contains("historical")) &
         (raw.year >= FIRST) & (raw.year <= LAST)].copy()
# unemployment ends in 2024: carry the last known rate forward to 2025 (an assumption)
df = df.sort_values(["region", "year"])
df["unemployment_rate_pct"] = df.groupby("region").unemployment_rate_pct.ffill()
# add an empty TARGET-year row for every region that has 2025 data
last = df[df.year == LAST].dropna(subset=["wage_median"])
new = pd.DataFrame({"region": last.region.values, "year": TARGET})
df = pd.concat([df, new], ignore_index=True).sort_values(["region", "year"])
df["cpi_nat"] = df.year.map(cpi_nat)
df["real_wage"] = df.wage_median / (df.cpi_nat / 100)
df["grp_pc"] = df.grp_mln_tenge * 1e6 / df.population
pc = lambda s: s.pct_change(fill_method=None)
g = df.groupby("region")
df["y"] = g.real_wage.transform(pc); df["nom_g"] = g.wage_median.transform(pc)
df["grp_g"] = g.grp_pc.transform(pc); df["pop_g"] = g.population.transform(pc)
nat = (raw[raw.region == NATIONAL].set_index("year").wage_median.div(cpi_nat / 100)
       ).pct_change(fill_method=None).rename("nat_g")
nat = nat.reindex(range(int(nat.index.min()), TARGET + 1))
g = df.groupby("region")
X = pd.DataFrame({"region": df.region, "year": df.year, "y": df.y, "y_lag1": g.y.shift(1),
    "nom_g_lag1": g.nom_g.shift(1), "unemp_lag1": g.unemployment_rate_pct.shift(1),
    "grp_g_lag1": g.grp_g.shift(1), "pop_g_lag1": g.pop_g.shift(1)}).merge(
    nat.shift(1).rename("nat_g_lag1"), left_on="year", right_index=True, how="left")
X["gap_lag1"] = X.y_lag1 - X.nat_g_lag1
train = X[X.year <= LAST].dropna(subset=["y"] + FEATS).reset_index(drop=True)
test = X[X.year == TARGET].copy()
miss = test[test[FEATS].isna().any(axis=1)].region.tolist()
test = test.dropna(subset=FEATS).reset_index(drop=True)
print(f"Training rows: {len(train)} (target years {train.year.min()}-{train.year.max()}), "
      f"regions forecast: {len(test)}" + (f", skipped (missing predictors): {miss}" if miss else ""))

def tune(pipe, grid):
    yrs = sorted(train.year.unique())
    cv = PredefinedSplit(np.where(train.year.isin(yrs[-2:]), train.year, -1))
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="neg_root_mean_squared_error"); gs.fit(train[FEATS], train.y)
    return gs.best_estimator_
svr = tune(make_pipeline(StandardScaler(), SVR(kernel="rbf")),
    {"svr__C": [0.1, 1, 10], "svr__epsilon": [0.005, 0.01, 0.02], "svr__gamma": ["scale", 0.05, 0.2]})
gb = tune(GradientBoostingRegressor(subsample=0.8, random_state=0),
    {"n_estimators": [50, 150], "learning_rate": [0.03, 0.1], "max_depth": [1, 2, 3], "min_samples_leaf": [3, 6]})
rf = tune(RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
    {"max_depth": [2, 3, 5, None], "min_samples_leaf": [3, 6, 10], "max_features": [0.33, 0.6, 1.0]})
ar = LinearRegression().fit(train[["y_lag1"]], train.y)

out = test[["region"]].copy()
out["SVR"] = svr.predict(test[FEATS]); out["GB"] = gb.predict(test[FEATS]); out["RF"] = rf.predict(test[FEATS])
out["ML_average"] = out[["SVR", "GB", "RF"]].mean(axis=1)
out["AR1"] = ar.predict(test[["y_lag1"]])
out["last_own_growth"] = test.y_lag1.values
out["last_national_growth"] = test.nat_g_lag1.values
out["ML_vs_regional_mean"] = out.ML_average - out.ML_average.mean()
out = out.sort_values("ML_average").reset_index(drop=True)
out.to_csv("forecast_2026.csv", index=False)
pd.options.display.float_format = "{:.3f}".format
print(f"\nForecast real median wage growth for {TARGET} (fraction; 0.05 = 5%), lowest first")
print(out.to_string(index=False))
print(f"\nSpread across regions (std of ML_average): {out.ML_average.std():.3f}")
print("Rough error scale: the pooled out-of-sample RMSE in 2023-2025 was about 0.07 (0.04-0.05 in 2023-2024, 0.10 in 2025).")

# --- download the file automatically when running in Colab
try:
    from google.colab import files
    files.download("forecast_2026.csv")
except ImportError:
    print("Saved forecast_2026.csv in the current folder.")

Training rows: 103 (target years 2020-2025), regions forecast: 20

Forecast real median wage growth for 2026 (fraction; 0.05 = 5%), lowest first
              region    SVR    GB    RF  ML_average   AR1  last_own_growth  last_national_growth  ML_vs_regional_mean
              Ulytau -0.074 0.131 0.103       0.053 0.079            0.016                -0.002               -0.037
         Astana city  0.001 0.129 0.088       0.073 0.071            0.060                -0.002               -0.018
           Kyzylorda -0.002 0.129 0.093       0.073 0.077            0.023                -0.002               -0.017
              Akmola -0.001 0.131 0.100       0.077 0.076            0.029                -0.002               -0.014
         Almaty city  0.018 0.129 0.093       0.080 0.074            0.042                -0.002               -0.010
             Zhambyl  0.021 0.130 0.096       0.082 0.074            0.042                -0.002               -0.008
            Kostanay  0.017 0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
"""Direct multi-year forecast (2026-2030) of real median wage growth by region: SVR, gradient boosting, random forest.
For each horizon h = 1..5 a separate model learns from region-years s the link between what was known in year s
and real wage growth in year s+h; it is then applied to the 2025 data to forecast 2025+h."""
import os, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, PredefinedSplit

PATH = "extended_data_6.xlsx"
if not os.path.exists(PATH):
    found = glob.glob("*extended*data*6*.xlsx") + glob.glob("/content/*extended*data*6*.xlsx")
    if not found: raise FileNotFoundError("Upload extended_data_6.xlsx and set PATH above.")
    PATH = found[0]
FIRST, LAST, HORIZONS = 2018, 2025, [1, 2, 3, 4, 5]
NATIONAL = "Republic of Kazakhstan"
FEATS = ["y", "nom_g", "unemp", "grp_g", "pop_g", "nat_g", "gap"]     # all measured in the origin year s

raw = pd.read_excel(PATH, sheet_name="Master_Panel")
for c in ["wage_median", "cpi_index", "unemployment_rate_pct", "grp_mln_tenge", "population"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
cpi_nat = raw[raw.region == NATIONAL].set_index("year")["cpi_index"]
df = raw[(raw.region != NATIONAL) & (~raw.region.str.contains("historical")) &
         (raw.year >= FIRST) & (raw.year <= LAST)].sort_values(["region", "year"]).copy()
df["unemp"] = df.groupby("region").unemployment_rate_pct.ffill()      # 2025 = 2024 rate (assumption)
df["real_wage"] = df.wage_median / (df.year.map(cpi_nat) / 100)
df["grp_pc"] = df.grp_mln_tenge * 1e6 / df.population
pc = lambda s: s.pct_change(fill_method=None)
g = df.groupby("region")
df["y"] = g.real_wage.transform(pc); df["nom_g"] = g.wage_median.transform(pc)
df["grp_g"] = g.grp_pc.transform(pc); df["pop_g"] = g.population.transform(pc)
nat = (raw[raw.region == NATIONAL].set_index("year").wage_median.div(cpi_nat / 100)).pct_change(fill_method=None)
df["nat_g"] = df.year.map(nat); df["gap"] = df.y - df.nat_g
g = df.groupby("region")

def tune_or_default(train, kind):
    yrs = sorted(train.s.unique())
    if kind == "svr":
        pipe = make_pipeline(StandardScaler(), SVR(kernel="rbf"))
        grid = {"svr__C": [0.1, 1, 10], "svr__epsilon": [0.005, 0.01, 0.02], "svr__gamma": ["scale", 0.05, 0.2]}
        default = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=1, epsilon=0.01))
    elif kind == "gb":
        pipe = GradientBoostingRegressor(subsample=0.8, random_state=0)
        grid = {"n_estimators": [50, 150], "learning_rate": [0.03, 0.1], "max_depth": [1, 2, 3], "min_samples_leaf": [3, 6]}
        default = GradientBoostingRegressor(n_estimators=50, learning_rate=0.05, max_depth=2, min_samples_leaf=6, subsample=0.8, random_state=0)
    else:
        pipe = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)
        grid = {"max_depth": [2, 3, 5, None], "min_samples_leaf": [3, 6, 10], "max_features": [0.33, 0.6, 1.0]}
        default = RandomForestRegressor(n_estimators=300, max_depth=3, min_samples_leaf=6, max_features=0.6, random_state=0, n_jobs=-1)
    if len(yrs) >= 4:      # enough origin years to hold out the last two for tuning
        cv = PredefinedSplit(np.where(train.s.isin(yrs[-2:]), train.s, -1))
        gs = GridSearchCV(pipe, grid, cv=cv, scoring="neg_root_mean_squared_error").fit(train[FEATS], train.target)
        return gs.best_estimator_, "tuned"
    return default.fit(train[FEATS], train.target), "default settings (too few years to tune)"

origin = df[df.year == LAST].dropna(subset=FEATS)
out = []
for h in HORIZONS:
    d = df.copy(); d["s"] = d.year; d["target"] = g.y.shift(-h)
    train = d[(d.s <= LAST - h)].dropna(subset=FEATS + ["target"])
    preds = {}
    note = ""
    for k in ["svr", "gb", "rf"]:
        mdl, note = tune_or_default(train, k)
        preds[k.upper()] = mdl.predict(origin[FEATS])
    print(f"h={h} ({LAST + h}): {len(train)} training region-years from {train.s.nunique()} origin years, {note}")
    r = pd.DataFrame({"region": origin.region.values, "horizon": h, "year": LAST + h, **preds})
    r["ML_average"] = r[["SVR", "GB", "RF"]].mean(axis=1)
    r["mean_of_training_growth"] = train.target.mean()
    r["last_national_growth"] = origin.nat_g.values
    r["n_train"] = len(train)
    out.append(r)
out = pd.concat(out, ignore_index=True)
out.to_csv("forecast_2026_2030.csv", index=False)
pd.options.display.float_format = "{:.3f}".format
print("\nAverage across the 20 regions, by year (fraction; 0.05 = 5%)")
print(out.groupby("year")[["SVR", "GB", "RF", "ML_average", "mean_of_training_growth", "last_national_growth"]].mean().to_string())
print("\nSpread across regions (std of ML_average), by year")
print(out.groupby("year").ML_average.std().to_string())
print("\nRegion forecasts, ML average")
print(out.pivot(index="region", columns="year", values="ML_average").to_string())
print("\nFor context, national real wage growth by year:", {int(y): round(float(v), 3) for y, v in nat.dropna().items()})

# --- save the file automatically: download it to your computer when running in Colab
try:
    from google.colab import files
    files.download("forecast_2026_2030.csv")
except ImportError:
    print("\nSaved forecast_2026_2030.csv in the current folder.")

h=1 (2026): 103 training region-years from 6 origin years, tuned
h=2 (2027): 83 training region-years from 5 origin years, tuned
h=3 (2028): 66 training region-years from 4 origin years, tuned
h=4 (2029): 49 training region-years from 3 origin years, default settings (too few years to tune)
h=5 (2030): 32 training region-years from 2 origin years, default settings (too few years to tune)

Average across the 20 regions, by year (fraction; 0.05 = 5%)
       SVR    GB    RF  ML_average  mean_of_training_growth  last_national_growth
year                                                                             
2026 0.021 0.133 0.117       0.090                    0.069                -0.002
2027 0.071 0.052 0.057       0.060                    0.051                -0.002
2028 0.076 0.048 0.055       0.060                    0.049                -0.002
2029 0.064 0.053 0.053       0.057                    0.038                -0.002
2030 0.046 0.046 0.045       0.046                    0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>